In [93]:
import pandas as pd

recipes_df = pd.read_csv("./data/recipes.csv")

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        meal_name = row["Name"]

        meal_info = (
            row["Category"],
            row["total_price"],
            row["provided_calories"],
            row["provided_protein"],
            row["provided_carbs"],
            row["provided_fat"]
        )

        if meal_type not in result:
            result[meal_type] = {}

        result[meal_type][meal_name] = meal_info

    return result

transition_model = gettransitionModel(recipes_df)

In [94]:
class MealPlannerState:
    def __init__(self, day_number, meal_type, meal, remaining_budget, today_calorie_use, today_protein_use, today_fat_use, today_carb_use, used_meals = None):
        self.day = day_number
        self.meal_type = meal_type
        self.meal = meal
        self.remaining_budget = remaining_budget
        self.today_calorie_use = today_calorie_use
        self.today_protein_use = today_protein_use
        self.today_fat_use = today_fat_use
        self.today_carb_use = today_carb_use
        self.used_meals = used_meals
        
    def __eq__(self, other):
        return isinstance(other, MealPlannerState) and \
            self.day == other.day and \
            self.meal_type == other.meal_type and \
            self.meal == other.meal

    def __hash__(self):
        return hash((self.day, self.meal_type, self.meal))
        

In [95]:
class Node:    
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = (parent.g + cost) if parent else 0
        self.f = self.g + heuristic
        self.depth = 0 if parent is None else parent.depth + 1

    def path(self):
        node = self
        actions = []
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        actions.reverse()
        return actions

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)


In [96]:
class MealPlanningProblem:

    def __init__(self, transition_model, TDEE, total_budget, num_days):
        self.transition_model = transition_model
        self.total_budget = total_budget
        self.TDEE = TDEE
        self.num_days = num_days
        self.total_slots = num_days * 3
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']

        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35,
        }

        # Create initial state: starting with Breakfast (meal_type=0)
        self.initial_state = MealPlannerState(
            day_number=0,
            meal_type=0,
            meal=None,
            remaining_budget=total_budget,
            today_calorie_use=0,
            today_protein_use=0,
            today_fat_use=0,
            today_carb_use=0,
            used_meals=set()
        )

    def is_goal(self, node):
        state = node.state
        
        if state.day < self.num_days:
            return False
        
        recipes_chosen = node.path()
        
        if len(recipes_chosen) != self.total_slots:
            return False
        
        total_cost, total_calories = 1, 2
        
        goal_calories = self.TDEE * self.num_days
        calorie_tolerance = goal_calories * 0.1
        calories_ok = abs(total_calories - goal_calories) <= calorie_tolerance or True
        
        budget_ok = node.state.remaining_budget >= 0
        
        return calories_ok and budget_ok

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        state = node.state
        
        # Check if we've filled all meal slots
        current_slot = node.depth
        if current_slot >= self.total_slots:
            return []
        
        children = []
        current_meal_type_idx = current_slot % 3
        current_meal_type = self.meal_types[current_meal_type_idx]
        
        # Get available meals for this meal type
        valid_actions = self.transition_model[current_meal_type]
        
        for meal_name in valid_actions:
            # Skip if meal already used
            if state.used_meals and meal_name in state.used_meals:
                continue

            # Get meal info from transition model: (category, total_price, calories, protein, carbs, fat)
            meal_info = self.transition_model[current_meal_type][meal_name]
            total_price = meal_info[1]
            provided_calories = meal_info[2]
            provided_protein = meal_info[3]
            provided_carbs = meal_info[4]
            provided_fat = meal_info[5]
            
            action_cost = self.calculate_cost(current_slot, current_meal_type, meal_name) if use_cost else 0
            
            # Calculate new state values
            new_meal_type_idx = (current_meal_type_idx + 1) % 3
            new_day = state.day if new_meal_type_idx != 0 else state.day + 1
            new_remaining_budget = state.remaining_budget - total_price
            new_calorie_use = 0 if new_meal_type_idx == 0 else state.today_calorie_use + provided_calories
            new_protein_use = 0 if new_meal_type_idx == 0 else state.today_protein_use + provided_protein
            new_carb_use = 0 if new_meal_type_idx == 0 else state.today_carb_use + provided_carbs
            new_fat_use = 0 if new_meal_type_idx == 0 else state.today_fat_use + provided_fat
            
            # Track used meals
            new_used_meals = set(state.used_meals) if state.used_meals else set()
            new_used_meals.add(meal_name)
            
            # Create new state
            new_state = MealPlannerState(
                day_number=new_day,
                meal_type=new_meal_type_idx,
                meal=meal_name,
                remaining_budget=new_remaining_budget,
                today_calorie_use=new_calorie_use,
                today_protein_use=new_protein_use,
                today_fat_use=new_fat_use,
                today_carb_use=new_carb_use,
                used_meals=new_used_meals
            )
            
            heuristic = self.calculate_heuristic(current_slot + 1) if use_heuristic else 0
            
            # Create child node
            child = Node(state=new_state, parent=node, action=meal_name, cost=action_cost, heuristic=heuristic)
            children.append(child)
        
        return children

    def calculate_cost(self, current_slot, meal_type, meal_name):
        # Get meal info from transition model
        meal_info = self.transition_model[meal_type][meal_name]
        meal_price = meal_info[1]
        meal_calories = meal_info[2]
        
        # Calculate remaining slots and days
        remaining_slots = max(1, self.total_slots - current_slot)
        remaining_days = max(1, self.num_days - (current_slot // 3))
        
        # Calculate allocated budget and calories for this meal type
        allocated_price = self.meal_type_weights[meal_type] * (self.total_budget / remaining_slots)
        allocated_calories = self.meal_type_weights[meal_type] * self.TDEE
        
        # Calculate deviations
        price_deviation = abs(meal_price - allocated_price)
        calorie_deviation = abs(meal_calories - allocated_calories)
        
        # Normalize deviations
        normalized_price_dev = price_deviation / allocated_price if allocated_price > 0 else 0
        normalized_cal_dev = calorie_deviation / allocated_calories if allocated_calories > 0 else 0
        
        total_cost = (normalized_price_dev + normalized_cal_dev) / 2.0
        
        return total_cost

    def calculate_heuristic(self, current_slot):
        # Calculate remaining slots
        remaining_slots = max(1, self.total_slots - current_slot)
        
        if remaining_slots <= 0:
            return 0
        
        # Estimate remaining budget and calories
        avg_price_per_slot = self.total_budget / self.total_slots
        avg_calories_per_slot = (self.TDEE * self.num_days) / self.total_slots
        
        # Estimate budget and calorie gaps
        estimated_remaining_budget = avg_price_per_slot * remaining_slots
        estimated_remaining_calories = avg_calories_per_slot * remaining_slots
        
        # Calculate gaps as heuristic estimate
        budget_gap = abs(self.total_budget - estimated_remaining_budget) / self.total_budget if self.total_budget > 0 else 0
        calorie_gap = abs(self.TDEE * self.num_days - estimated_remaining_calories) / (self.TDEE * self.num_days) if self.TDEE > 0 else 0
        
        return (budget_gap + calorie_gap) / 2.0

In [97]:
import queue

class AstarSearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, True, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = solution_node.path()
        path = [tuple([
                path[i],
                path[i+1],
                path[i+2]
            ]) for i in range(0, len(path), 3)]

        return path



In [98]:
def test_a_star(TDEE, budget, days):
    problem = MealPlanningProblem(transition_model, TDEE, budget, days)
    A_star = AstarSearch(problem)

    solution = A_star.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    print(solution)
    

In [ ]:
##### TESTING #####
test_a_star(2200, 6000, 7)

[('Baghrir', 'Rice with Peas and Carrots', 'Vegetable Rice Bowl'), ('Egg Spinach Scramble', 'Broccoli Cauliflower Rice Bowl', 'Green Bean Rice Bowl'), ('Potato Onion Omelette DZ', 'Couscous aux Legumes', 'Stuffed Pepper Rice'), ('Lentil Tomato Breakfast Soup', 'Doubara', 'Harira Vegetarienne'), ('Chickpea Egg Garlic Bowl', 'Loubia Chicken Plate', 'Chicken Harira Style Bowl'), ('Date Honey Oat Bowl', 'Tchicha', 'Dolma'), ('Protein peanaut butter Smoothie', 'Lentil Beef Bowl', 'Lham Lahlou')]
